<a href="https://colab.research.google.com/github/DArturo-LR/Analizador-de-Datos-Demogr-ficos/blob/main/Analizador_de_datos_demograficos2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
pip install ucimlrepo

In [18]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features
y = adult.data.targets

# metadata
print(adult.metadata)

# variable information
print(adult.variables)


{'uci_id': 2, 'name': 'Adult', 'repository_url': 'https://archive.ics.uci.edu/dataset/2/adult', 'data_url': 'https://archive.ics.uci.edu/static/public/2/data.csv', 'abstract': 'Predict whether annual income of an individual exceeds $50K/yr based on census data. Also known as "Census Income" dataset. ', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Tue Sep 24 2024', 'dataset_doi': '10.24432/C5XW20', 'creators': ['Barry Becker', 'Ronny Kohavi'], 'intro_paper': None, 'additional_info': {'summary': "Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the fol

In [19]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

def calculate_demographic_data(print_data=True):
    # 1. Descargar el dataset usando la librería oficial
    adult = fetch_ucirepo(id=2)

    X = adult.data.features
    y = adult.data.targets

    # 2. Combinar X e y en un único DataFrame
    df = pd.concat([X, y], axis=1)

    # Limpiar espacios en blanco en las columnas de texto
    df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # ------------------ CÁLCULOS ------------------

    # 1. ¿Cuántas personas de cada raza están representadas?
    conteo_razas = df['race'].value_counts()

    # 2. ¿Cuál es la edad promedio de los hombres?
    edad_promedio_hombres = round(df[df['sex'] == 'Male']['age'].mean(), 1)

    # 3. ¿Cuál es el porcentaje de personas que tienen un grado de licenciatura (Bachelors)?
    porcentaje_licenciatura = round((df['education'] == 'Bachelors').sum() / len(df) * 100, 1)

    # 4. Educación avanzada (Bachelors, Masters, Doctorate)
    educacion_avanzada = df['education'].isin(['Bachelors', 'Masters', 'Doctorate'])
    educacion_basica = ~educacion_avanzada

    # Porcentaje con educación avanzada que gana más de 50k
    df_avanzada = df[educacion_avanzada]
    porcentaje_avanzada_ricos = round((df_avanzada['income'].isin(['>50K', '>50K.'])).sum() / len(df_avanzada) * 100, 1)

    # 5. Sin educación avanzada que gana más de 50k
    df_basica = df[educacion_basica]
    porcentaje_basica_ricos = round((df_basica['income'].isin(['>50K', '>50K.'])).sum() / len(df_basica) * 100, 1)

    # 6. Mínimo de horas semanales trabajadas
    min_horas_trabajo = df['hours-per-week'].min()

    # 7. Porcentaje de personas que trabajando el mínimo ganan más de 50k
    num_min_trabajadores = df[df['hours-per-week'] == min_horas_trabajo]
    # CORREGIDO: Ahora usa 'num_min_trabajadores' en lugar de la variable en inglés
    porcentaje_min_ricos = round((num_min_trabajadores['income'].isin(['>50K', '>50K.'])).sum() / len(num_min_trabajadores) * 100, 1)

    # 8. País con mayor porcentaje de altos ingresos y su porcentaje
    conteos_por_pais = df['native-country'].value_counts()
    ricos_por_pais = df[df['income'].isin(['>50K', '>50K.'])]['native-country'].value_counts()

    porcentaje_ricos_pais = (ricos_por_pais / conteos_por_pais) * 100

    pais_mayores_ingresos = porcentaje_ricos_pais.idxmax()
    porcentaje_mayores_ingresos_pais = round(porcentaje_ricos_pais.max(), 1)

    # 9. Ocupación más popular para >50K en India
    india_ricos = df[(df['native-country'] == 'India') & (df['income'].isin(['>50K', '>50K.']))]
    ocupacion_top_india = india_ricos['occupation'].value_counts().idxmax()

    # ------------------ IMPRESIÓN DE RESULTADOS ------------------
    if print_data:
        print("Cantidad de personas por cada raza:\n", conteo_razas)
        print("Edad promedio de los hombres:", edad_promedio_hombres)
        print(f"Porcentaje con grado de Licenciatura: {porcentaje_licenciatura}%")
        print(f"Porcentaje con educación avanzada que gana >50K: {porcentaje_avanzada_ricos}%")
        print(f"Porcentaje sin educación avanzada que gana >50K: {porcentaje_basica_ricos}%")
        print(f"Tiempo mínimo de trabajo: {min_horas_trabajo} horas/semana")
        print(f"Porcentaje de altos ingresos entre los que trabajan el mínimo de horas: {porcentaje_min_ricos}%")
        print("País con el porcentaje más alto de altos ingresos:", pais_mayores_ingresos)
        print(f"Porcentaje más alto de ingresos por país: {porcentaje_mayores_ingresos_pais}%")
        print("Ocupación más popular para altos ingresos en India:", ocupacion_top_india)

    return {
        'conteo_razas': conteo_razas,
        'edad_promedio_hombres': edad_promedio_hombres,
        'porcentaje_licenciatura': porcentaje_licenciatura,
        'porcentaje_avanzada_ricos': porcentaje_avanzada_ricos,
        'porcentaje_basica_ricos': porcentaje_basica_ricos,
        'min_horas_trabajo': min_horas_trabajo,
        'porcentaje_min_ricos': porcentaje_min_ricos,
        'pais_mayores_ingresos': pais_mayores_ingresos,
        'porcentaje_mayores_ingresos_pais': porcentaje_mayores_ingresos_pais,
        'ocupacion_top_india': ocupacion_top_india
    }

In [20]:
# EJECUTA ESTA CELDA PARA VER LOS RESULTADOS
print("--- EJECUTANDO ANÁLISIS DEMOGRÁFICO ---")
resultados = calculate_demographic_data()

--- EJECUTANDO ANÁLISIS DEMOGRÁFICO ---
Cantidad de personas por cada raza:
 race
White                 41762
Black                  4685
Asian-Pac-Islander     1519
Amer-Indian-Eskimo      470
Other                   406
Name: count, dtype: int64
Edad promedio de los hombres: 39.5
Porcentaje con grado de Licenciatura: 16.4%
Porcentaje con educación avanzada que gana >50K: 46.1%
Porcentaje sin educación avanzada que gana >50K: 17.3%
Tiempo mínimo de trabajo: 1 horas/semana
Porcentaje de altos ingresos entre los que trabajan el mínimo de horas: 11.1%
País con el porcentaje más alto de altos ingresos: France
Porcentaje más alto de ingresos por país: 42.1%
Ocupación más popular para altos ingresos en India: Prof-specialty
